In [ ]:
from pathlib import Path

import pandas as pd

EXPERIMENT = "gulf_stream_pigment_influencers_20241001_20251231"
repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
gold = pd.read_parquet(repo_root / "data" / EXPERIMENT / "gold" / "eddy_pigment_table.parquet")
gold

In [ ]:
# Mean eddy concentration, eddy/background ratio, and log-ratio per pigment,
# pooled by polarity across all eddy-days.
import numpy as np

pigments = [c.removeprefix("eddy_mean_") for c in gold.columns if c.startswith("eddy_mean_")]
pol = gold["polarity"].map({0: "anticyclone", 1: "cyclone"})
eddy = gold[[f"eddy_mean_{p}" for p in pigments]].set_axis(pigments, axis=1)
bg = gold[[f"bg_mean_{p}" for p in pigments]].set_axis(pigments, axis=1)
log_ratio = gold[[f"log_ratio_{p}" for p in pigments]].set_axis(pigments, axis=1).replace([np.inf, -np.inf], np.nan)

metrics = {"eddy_mean": eddy, "ratio": eddy / bg, "log_ratio": log_ratio}
summary = pd.concat({name: frame.groupby(pol).mean().T for name, frame in metrics.items()}, axis=1)
summary.round(3)